# Densenet121 Multiclass training with CONNIE image dataset

In [1]:
%run ./../notebook_init.py

import os
import torch
import optuna
import mlflow

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from torch.utils.data import Subset
from itertools import product
from pathlib import Path
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix

from core import DATA_FOLDER, RESULTS_FOLDER

from scripts.connie_training_utils import ModelTraining, TransformedSubset, \
    Seed, get_test_transform, IMG_SIZE, get_train_transform,\
    densenet121_model, NPYFolderDataset

Load file paths and set the computation device to GPU if available; otherwise, use CPU, and initialize the random seed

In [2]:
#train_data = os.path.join(DATA_FOLDER, "train_data_png_full")
train_data = os.path.join(DATA_FOLDER, "train_data_npy_full")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
seed = Seed()

cuda:0


In [3]:
trainval_dataset = NPYFolderDataset(train_data)
trainval_set = Subset(trainval_dataset, list(range(len(trainval_dataset))))

Compute dataset mean and standard deviation, then define training and test transforms with normalization

In [4]:
#basic_transform = transforms.Compose([
#    transforms.Resize(IMG_SIZE),
#    transforms.ToTensor()
#])
#
## Load dataset without transform
#full_dataset_transform = datasets.ImageFolder(train_data,
#                                              transform=basic_transform)
#
#mean, std = calculate_mean_std(full_dataset_transform)
#test_transform = get_test_transform(mean, std)
#train_transform = get_train_transform(mean, std)

Split train + validation and test set

In [5]:
#trainval_dataset = datasets.ImageFolder(train_data)
#trainval_set = Subset(trainval_dataset, list(range(len(trainval_dataset))))

## Training with K-fold

In [6]:
# Define your parameter grid
param_grid = {
    "learning_rate": [5e-4],
    'weight_decay': [5e-5],
    "step_size": [10],
    "gamma": [0.5]
}

# Create all combinations
grid = list(product(
    param_grid["learning_rate"],
    param_grid["weight_decay"],
    param_grid["step_size"],
    param_grid["gamma"]
))

k_folds = 5
num_epochs = 100

class_idx_map = trainval_dataset.class_to_idx
print("Classes index:", class_idx_map)

Classes index: {'Blob': 0, 'Diffusion Hit': 1, 'Electron': 2, 'Muon': 3, 'Others': 4}


## Hyperparameters Tuning

In [7]:
from pathlib import Path
mlflow.set_tracking_uri(Path(DATA_FOLDER) / "mlruns")
#mlflow.set_tracking_uri(os.path.join(DATA_FOLDER, "mlruns"))

def objective_densenet121(trial):
    k_folds = 5
    num_epochs = 100
    model_training = ModelTraining()

    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    wd = trial.suggest_float("wd", 1e-5, 1e-2, log=True)
    step_size = trial.suggest_int("step", 5, 50)
    gamma = trial.suggest_float("gamma", 0.1, 0.9)

    hyperparam = {"lr": lr, "wd": wd, "step": step_size, "gamma": gamma}

    with mlflow.start_run(nested=True, run_name=f"DenseNet_trial_{trial.number}"):
        mlflow.set_tag("model_type", "DenseNet-121")
        mlflow.set_tag("kfold_splits", k_folds)
        mlflow.log_params(hyperparam)

        metrics = model_training.train_model_kfold_multiclass(
            device=device,
            dataset=trainval_set,
            num_epochs=num_epochs,
            k_folds=k_folds,
            seed=seed,
            model=densenet121_model,
            hyperparam=hyperparam
        )

        mlflow.log_metrics({
            "mean_train_accuracy": metrics["mean_train_accuracy"],
            "std_train_accuracy": metrics["std_train_accuracy"],
            "mean_train_loss": metrics["mean_train_loss"],
        
            "mean_val_accuracy": metrics["mean_val_accuracy"],
            "std_val_accuracy": metrics["std_val_accuracy"],
            "mean_val_loss": metrics["mean_val_loss"],
            "std_val_loss": metrics["std_val_loss"],
        
            "mean_precision": metrics["mean_precision"],
            "std_precision": metrics["std_precision"],
            "mean_recall": metrics["mean_recall"],
            "std_recall": metrics["std_recall"],
            "mean_f1_macro": metrics["mean_f1_macro"],
            "std_f1_macro": metrics["std_f1_macro"]
        })
        best_epochs = metrics.get("best_epochs_per_fold")
        if best_epochs is not None:
            mlflow.log_metric("best_epoch_mean", float(np.mean(best_epochs)))
            mlflow.log_metric("best_epoch_std", float(np.std(best_epochs)))
            mlflow.log_metric("best_epoch_median", float(np.median(best_epochs)))
        
        mlflow.log_dict(metrics, "full_metrics.json")

        all_val_true = metrics["all_val_true"]
        all_val_preds = metrics["all_val_preds"]

        metrics_dir = os.path.join(RESULTS_FOLDER,
                                   "metrics_densenet121_multiclass_no_alpha")
        os.makedirs(metrics_dir, exist_ok=True)

        class_names = trainval_set.dataset.classes

        # === Classification report ===
        report = classification_report(
            all_val_true,
            all_val_preds,
            target_names=class_names,
            output_dict=True,
            zero_division=0
        )
        report_df = pd.DataFrame(report).transpose()

        report_path = os.path.join(metrics_dir,
                                   f"trial_{trial.number}_classification_report.csv")
        report_df.to_csv(report_path)
        mlflow.log_artifact(report_path)

        # === Confusion matrix ===
        cm = confusion_matrix(
            all_val_true,
            all_val_preds,
            labels=np.arange(len(class_names))
        )
        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names,
            ax=ax
        )
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        fig.tight_layout()

        cm_path = os.path.join(metrics_dir,
                               f"trial_{trial.number}_confusion_matrix.png")
        fig.savefig(cm_path)
        mlflow.log_artifact(cm_path)
        plt.close(fig)

        return metrics["mean_f1_macro"]

In [8]:
class_idx_map = trainval_set.dataset.class_to_idx

   
mlflow.set_experiment(f"Tuning_Densenet121_multiclass_no_alpha_fixed_2")
study_densenet121 = optuna.create_study(direction="maximize")
study_densenet121.optimize(lambda trial: objective_densenet121(trial),
                           n_trials=100)

print(f"Best trials for Densenet121:")
for i, t in enumerate(study_densenet121.best_trials):
    print(f"Trial #{t.number}")
    print(f"  Values (Val Accuracy, Val Loss): {t.values}")
    print(f"  Params: ")
    for key, value in t.params.items():
        print(f"    {key}: {value}")

2026/04/08 14:43:47 INFO mlflow.tracking.fluent: Experiment with name 'Tuning_Densenet121_multiclass_no_alpha_fixed_2' does not exist. Creating a new experiment.
[I 2026-04-08 14:43:47,684] A new study created in memory with name: no-name-c6ca92e4-fde1-414d-a0f1-5da50d062a41



==================== Fold 1/5 ====================

Epoch 1/100 — Fold 1
----------
Train Loss: 1.0551 Acc: 0.5432
Val Loss: 1.3199 Acc: 0.7179
Val Precision: 0.3884 Recall: 0.2075 F1: 0.1864

Epoch 2/100 — Fold 1
----------
Train Loss: 0.6985 Acc: 0.7337
Val Loss: 1.4250 Acc: 0.6089
Val Precision: 0.5404 Recall: 0.7034 F1: 0.5637

Epoch 3/100 — Fold 1
----------
Train Loss: 0.7232 Acc: 0.7200
Val Loss: 0.7935 Acc: 0.6788
Val Precision: 0.6216 Recall: 0.7281 F1: 0.6113

Epoch 4/100 — Fold 1
----------
Train Loss: 0.7581 Acc: 0.6760
Val Loss: 0.5159 Acc: 0.8324
Val Precision: 0.6839 Recall: 0.7488 F1: 0.7109

Epoch 5/100 — Fold 1
----------
Train Loss: 0.6703 Acc: 0.7441
Val Loss: 0.6637 Acc: 0.8059
Val Precision: 0.4612 Recall: 0.5191 F1: 0.4731

Epoch 6/100 — Fold 1
----------
Train Loss: 0.6919 Acc: 0.7462
Val Loss: 0.7590 Acc: 0.6788
Val Precision: 0.6222 Recall: 0.7462 F1: 0.6429

Epoch 7/100 — Fold 1
----------
Train Loss: 0.6425 Acc: 0.7309
Val Loss: 1.5435 Acc: 0.2626
Val Preci

[I 2026-04-08 15:22:20,937] Trial 0 finished with value: 0.7973232985063192 and parameters: {'lr': 0.0016634034627173082, 'wd': 0.0026637587182019597, 'step': 18, 'gamma': 0.6109252004171569}. Best is trial 0 with value: 0.7973232985063192.



==================== Fold 1/5 ====================

Epoch 1/100 — Fold 1
----------
Train Loss: 0.7903 Acc: 0.6893
Val Loss: 1.7686 Acc: 0.3589
Val Precision: 0.2957 Recall: 0.4651 F1: 0.2570

Epoch 2/100 — Fold 1
----------
Train Loss: 0.5442 Acc: 0.8022
Val Loss: 0.6128 Acc: 0.7458
Val Precision: 0.6425 Recall: 0.7926 F1: 0.6848

Epoch 3/100 — Fold 1
----------
Train Loss: 0.4564 Acc: 0.8427
Val Loss: 0.6024 Acc: 0.7905
Val Precision: 0.6415 Recall: 0.8262 F1: 0.7057

Epoch 4/100 — Fold 1
----------
Train Loss: 0.5657 Acc: 0.8207
Val Loss: 1.0257 Acc: 0.5866
Val Precision: 0.5204 Recall: 0.7715 F1: 0.5629

Epoch 5/100 — Fold 1
----------
Train Loss: 0.4558 Acc: 0.8406
Val Loss: 2.1696 Acc: 0.4721
Val Precision: 0.5399 Recall: 0.6644 F1: 0.4996

Epoch 6/100 — Fold 1
----------
Train Loss: 0.4972 Acc: 0.8410
Val Loss: 0.3358 Acc: 0.8994
Val Precision: 0.7927 Recall: 0.8778 F1: 0.8270

Epoch 7/100 — Fold 1
----------
Train Loss: 0.3668 Acc: 0.8693
Val Loss: 0.3744 Acc: 0.8715
Val Preci

[I 2026-04-08 15:53:28,477] Trial 1 finished with value: 0.8469605075174578 and parameters: {'lr': 0.0005794074295923148, 'wd': 0.0012938753454387623, 'step': 49, 'gamma': 0.6700649241833321}. Best is trial 1 with value: 0.8469605075174578.


...


==================== Fold 1/5 ====================

Epoch 1/100 — Fold 1
----------
Train Loss: 0.7676 Acc: 0.6924
Val Loss: 0.6865 Acc: 0.7877
Val Precision: 0.6204 Recall: 0.6842 F1: 0.5864

Epoch 2/100 — Fold 1
----------
Train Loss: 0.5778 Acc: 0.8085
Val Loss: 2.8451 Acc: 0.3617
Val Precision: 0.4267 Recall: 0.6960 F1: 0.4049

Epoch 3/100 — Fold 1
----------
Train Loss: 0.5565 Acc: 0.8158
Val Loss: 0.5641 Acc: 0.7835
Val Precision: 0.6497 Recall: 0.8258 F1: 0.7110

Epoch 4/100 — Fold 1
----------
Train Loss: 0.4833 Acc: 0.8392
Val Loss: 0.4922 Acc: 0.8296
Val Precision: 0.5948 Recall: 0.7110 F1: 0.6262

Epoch 5/100 — Fold 1
----------
Train Loss: 0.4685 Acc: 0.8560
Val Loss: 0.5365 Acc: 0.8073
Val Precision: 0.6504 Recall: 0.8181 F1: 0.7084

Epoch 6/100 — Fold 1
----------
Train Loss: 0.4735 Acc: 0.8413
Val Loss: 0.3786 Acc: 0.8771
Val Precision: 0.7574 Recall: 0.8450 F1: 0.7939

Epoch 7/100 — Fold 1
----------
Train Loss: 0.5140 Acc: 0.8445
Val Loss: 0.3467 Acc: 0.8785
Val Preci

[I 2026-04-11 03:35:37,689] Trial 98 finished with value: 0.866150637990932 and parameters: {'lr': 0.0006012277291469664, 'wd': 0.00019299985183752016, 'step': 13, 'gamma': 0.5775691370476432}. Best is trial 43 with value: 0.8752391342310915.



==================== Fold 1/5 ====================

Epoch 1/100 — Fold 1
----------
Train Loss: 0.8248 Acc: 0.6760
Val Loss: 1.4795 Acc: 0.4539
Val Precision: 0.4505 Recall: 0.5020 F1: 0.3379

Epoch 2/100 — Fold 1
----------
Train Loss: 0.5151 Acc: 0.8259
Val Loss: 0.4125 Acc: 0.8506
Val Precision: 0.6992 Recall: 0.8681 F1: 0.7590

Epoch 3/100 — Fold 1
----------
Train Loss: 0.4638 Acc: 0.8445
Val Loss: 0.5409 Acc: 0.8198
Val Precision: 0.6802 Recall: 0.8243 F1: 0.7349

Epoch 4/100 — Fold 1
----------
Train Loss: 0.4325 Acc: 0.8476
Val Loss: 0.4262 Acc: 0.8617
Val Precision: 0.7343 Recall: 0.8804 F1: 0.7930

Epoch 5/100 — Fold 1
----------
Train Loss: 0.4466 Acc: 0.8623
Val Loss: 0.5767 Acc: 0.8226
Val Precision: 0.5353 Recall: 0.6290 F1: 0.5656

Epoch 6/100 — Fold 1
----------
Train Loss: 0.4750 Acc: 0.8417
Val Loss: 0.3980 Acc: 0.8631
Val Precision: 0.7101 Recall: 0.8655 F1: 0.7715

Epoch 7/100 — Fold 1
----------
Train Loss: 0.3894 Acc: 0.8469
Val Loss: 0.3348 Acc: 0.8883
Val Preci

[I 2026-04-11 04:05:33,072] Trial 99 finished with value: 0.8730022023172825 and parameters: {'lr': 0.0004149809635171603, 'wd': 2.1348611273859817e-05, 'step': 11, 'gamma': 0.5188523011371933}. Best is trial 43 with value: 0.8752391342310915.


Best trials for Densenet121:
Trial #43
  Values (Val Accuracy, Val Loss): [0.8752391342310915]
  Params: 
    lr: 0.0003544119262819953
    wd: 0.00432000936911502
    step: 15
    gamma: 0.3582896144401891
